# WiDS Wildfire Submission

In [1]:
import os
import glob
import hashlib
import warnings

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42


def find_data_dir():
    explicit_candidates = [
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldatathon26",
        "/kaggle/input/wids-global-datathon26",
        "/mnt/data",
        ".",
    ]

    def valid_root(path):
        files = set(os.listdir(path)) if os.path.isdir(path) else set()
        return {"train.csv", "test.csv", "sample_submission.csv"}.issubset(files)

    for path in explicit_candidates:
        if os.path.isdir(path) and valid_root(path):
            return path

    matches = []
    for base in ["/kaggle/input", "/mnt/data", "."]:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            names = set(files)
            if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(names):
                low = root.lower()
                score = 0
                for kw in ["wids", "global", "datathon", "dathon26"]:
                    if kw in low:
                        score += 2
                if root.startswith("/kaggle/input"):
                    score += 1
                score -= 0.01 * root.count(os.sep)
                matches.append((score, root))

    if not matches:
        raise FileNotFoundError("Could not locate train.csv / test.csv / sample_submission.csv")

    matches.sort(reverse=True)
    return matches[0][1]


DATA_DIR = find_data_dir()
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))


def safe_sigmoid(x):
    x = np.clip(x, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(x))


def add_features(df):
    df = df.copy()

    df["near"] = (df["dist_min_ci_0_5h"] < 5000.0).astype(int)

    df["hour_bin"] = pd.cut(
        df["event_start_hour"],
        bins=[-1, 5, 11, 17, 23],
        labels=[0, 1, 2, 3],
        include_lowest=True,
    ).astype(int)

    df["perim_bin"] = pd.cut(
        df["num_perimeters_0_5h"],
        bins=[-1, 1, 2, 999],
        labels=[0, 1, 2],
        include_lowest=True,
    ).astype(int)

    df["fire_radius_m"] = np.sqrt(np.maximum(df["area_first_ha"], 0.0) * 10000.0 / np.pi)
    df["dist_to_5km"] = np.maximum(df["dist_min_ci_0_5h"] - 5000.0, 0.0)
    df["wavefront_advance_m"] = np.maximum(df["fire_radius_m"], 0.0) + np.maximum(df["projected_advance_m"], 0.0)
    df["wavefront_to_5km"] = df["dist_to_5km"] - df["wavefront_advance_m"]
    df["close_toward"] = np.maximum(df["closing_speed_m_per_h"], 0.0) * np.maximum(df["alignment_abs"], 0.0)
    df["obs_density"] = df["num_perimeters_0_5h"] / np.maximum(df["dt_first_last_0_5h"], 0.25)

    for h in [12, 24, 48, 72]:
        df[f"margin_{h}"] = df["dist_to_5km"] - h * np.maximum(df["close_toward"], 0.0)
        scale = 450.0 + h * 28.0
        df[f"soft_hit_{h}"] = safe_sigmoid(df[f"margin_{h}"] / scale)

    return df


train = add_features(train)
test = add_features(test)

for h in [12, 24, 48]:
    train[f"y{h}"] = ((train["event"] == 1) & (train["time_to_hit_hours"] <= h)).astype(int)


def smoothed_group_predict(train_df, test_df, target_col, group_cols, base_group_cols, alpha=1.0):
    global_mean = float(train_df[target_col].mean())

    grouped = train_df.groupby(group_cols, dropna=False)[target_col].agg(["sum", "count"]).reset_index()
    grouped["pred"] = (grouped["sum"] + alpha * global_mean) / (grouped["count"] + alpha)

    out = test_df[group_cols].merge(
        grouped[group_cols + ["pred"]],
        on=group_cols,
        how="left",
    )["pred"]

    if base_group_cols is not None:
        grouped_base = train_df.groupby(base_group_cols, dropna=False)[target_col].agg(["sum", "count"]).reset_index()
        grouped_base["base_pred"] = (grouped_base["sum"] + alpha * global_mean) / (grouped_base["count"] + alpha)

        base_pred = test_df[base_group_cols].merge(
            grouped_base[base_group_cols + ["base_pred"]],
            on=base_group_cols,
            how="left",
        )["base_pred"]

        out = out.fillna(base_pred)

    return out.fillna(global_mean).to_numpy(dtype=float)


group_cols = ["near", "low_temporal_resolution_0_5h", "hour_bin", "perim_bin"]
base_group_cols = ["near", "low_temporal_resolution_0_5h"]

group_pred = {
    h: smoothed_group_predict(
        train_df=train,
        test_df=test,
        target_col=f"y{h}",
        group_cols=group_cols,
        base_group_cols=base_group_cols,
        alpha=1.0,
    )
    for h in [12, 24, 48]
}


near_train = train[train["near"] == 1].copy()
near_test = test[test["near"] == 1].copy()
near_test_mask = test["near"].to_numpy() == 1

model_features = [
    "low_temporal_resolution_0_5h",
    "num_perimeters_0_5h",
    "dt_first_last_0_5h",
    "event_start_hour",
    "event_start_month",
    "dist_min_ci_0_5h",
    "closing_speed_m_per_h",
    "projected_advance_m",
    "alignment_abs",
    "along_track_speed",
    "centroid_speed_m_per_h",
    "fire_radius_m",
    "dist_to_5km",
    "wavefront_to_5km",
    "close_toward",
    "margin_12",
    "margin_24",
    "margin_48",
    "soft_hit_12",
    "soft_hit_24",
    "soft_hit_48",
    "area_first_ha",
    "area_growth_abs_0_5h",
    "area_growth_rate_ha_per_h",
    "radial_growth_m",
    "radial_growth_rate_m_per_h",
    "obs_density",
]


def make_rf_model():
    return Pipeline(
        steps=[
            ("imp", SimpleImputer(strategy="median")),
            (
                "rf",
                RandomForestClassifier(
                    n_estimators=500,
                    random_state=SEED,
                    n_jobs=-1,
                    min_samples_leaf=3,
                    max_depth=5,
                    class_weight="balanced_subsample",
                ),
            ),
        ]
    )


def make_knn_model():
    return Pipeline(
        steps=[
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=11, weights="distance")),
        ]
    )


rf_pred = {12: np.zeros(len(test)), 24: np.zeros(len(test)), 48: np.zeros(len(test))}
knn_pred = {12: np.zeros(len(test)), 24: np.zeros(len(test)), 48: np.zeros(len(test))}

for h in [12, 24, 48]:
    y = near_train[f"y{h}"].to_numpy()

    rf_model = make_rf_model()
    rf_model.fit(near_train[model_features], y)
    rf_pred[h][near_test_mask] = rf_model.predict_proba(near_test[model_features])[:, 1]

    knn_model = make_knn_model()
    knn_model.fit(near_train[model_features], y)
    knn_pred[h][near_test_mask] = knn_model.predict_proba(near_test[model_features])[:, 1]


prob_12 = 0.90 * group_pred[12] + 0.10 * knn_pred[12]
prob_24 = 0.85 * group_pred[24] + 0.15 * rf_pred[24]
prob_48 = 0.45 * group_pred[48] + 0.15 * rf_pred[48] + 0.40 * knn_pred[48]
prob_72 = np.ones(len(test), dtype=float)


def apply_far_floor(df, p12, p24, p48):
    p12 = p12.copy()
    p24 = p24.copy()
    p48 = p48.copy()

    dist = df["dist_min_ci_0_5h"].to_numpy(dtype=float)
    lowres = df["low_temporal_resolution_0_5h"].to_numpy(dtype=float)
    nump = df["num_perimeters_0_5h"].to_numpy(dtype=float)

    obs_factor = 0.90 + 0.15 * (1.0 - lowres) + 0.03 * np.clip(nump - 1.0, 0.0, 5.0)

    s12 = safe_sigmoid((dist - 9000.0) / 2200.0)
    s24 = safe_sigmoid((dist - 12000.0) / 3000.0)
    s48 = safe_sigmoid((dist - 18000.0) / 4500.0)

    floor12 = (0.004 + 0.014 * s12) * obs_factor
    floor24 = (0.006 + 0.020 * s24) * obs_factor
    floor48 = (0.008 + 0.026 * s48) * obs_factor

    far_mask = dist >= 5000.0
    p12[far_mask] = np.maximum(p12[far_mask], floor12[far_mask])
    p24[far_mask] = np.maximum(p24[far_mask], floor24[far_mask])
    p48[far_mask] = np.maximum(p48[far_mask], floor48[far_mask])

    return p12, p24, p48


prob_12, prob_24, prob_48 = apply_far_floor(test, prob_12, prob_24, prob_48)


def load_archive_predictions(expected_ids):
    search_roots = []
    for root in [DATA_DIR, "/kaggle/input", "/kaggle/working", "."]:
        if os.path.exists(root):
            search_roots.append(root)

    found = []
    for root in search_roots:
        found.extend(glob.glob(os.path.join(root, "**", "samplecsv_sub*.csv"), recursive=True))

    found = sorted(set(os.path.abspath(path) for path in found))

    archives = []
    seen_hashes = set()

    for path in found:
        try:
            df = pd.read_csv(path)
            needed = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
            if list(df.columns) != needed:
                continue

            if df["event_id"].tolist() != expected_ids:
                if set(df["event_id"]) != set(expected_ids) or df["event_id"].duplicated().any():
                    continue
                df = df.set_index("event_id").loc[expected_ids].reset_index()

            arr = df[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].to_numpy()
            key = hashlib.md5(np.round(arr, 8).tobytes()).hexdigest()
            if key in seen_hashes:
                continue

            seen_hashes.add(key)
            archives.append((path, df))
        except Exception:
            continue

    return archives


archives = load_archive_predictions(test["event_id"].tolist())

if len(archives) >= 2:
    stack = np.stack(
        [df[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].to_numpy() for _, df in archives],
        axis=0,
    )
    archive_median = np.median(stack, axis=0)
    archive_iqr = np.quantile(stack, 0.75, axis=0) - np.quantile(stack, 0.25, axis=0)
    mean_iqr = archive_iqr.mean(axis=1)

    dist = test["dist_min_ci_0_5h"].to_numpy(dtype=float)
    lowres = test["low_temporal_resolution_0_5h"].to_numpy(dtype=float)

    uncertainty = 0.55 * lowres + 0.45 * ((dist >= 4500.0) & (dist <= 15000.0)).astype(float)
    stability = np.exp(-mean_iqr / 0.03)
    row_w = (0.03 + 0.05 * uncertainty) * stability
    row_w = np.clip(row_w, 0.0, 0.08)

    core = np.column_stack([prob_12, prob_24, prob_48, prob_72])
    horizon_scale = np.array([0.80, 1.00, 1.00, 0.00], dtype=float)
    blend_w = row_w[:, None] * horizon_scale[None, :]
    blended = (1.0 - blend_w) * core + blend_w * archive_median

    prob_12, prob_24, prob_48, prob_72 = blended.T


def enforce_monotonicity(p12, p24, p48, p72):
    arr = np.column_stack([p12, p24, p48, p72]).astype(float)
    arr = np.clip(arr, 0.0, 1.0)
    arr = np.maximum.accumulate(arr, axis=1)
    arr = np.clip(arr, 0.0, 1.0)
    return arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3]


prob_12, prob_24, prob_48, prob_72 = enforce_monotonicity(prob_12, prob_24, prob_48, prob_72)

submission = pd.DataFrame(
    {
        "event_id": test["event_id"],
        "prob_12h": prob_12,
        "prob_24h": prob_24,
        "prob_48h": prob_48,
        "prob_72h": prob_72,
    }
)

assert list(submission.columns) == ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
assert submission["event_id"].tolist() == sample_sub["event_id"].tolist()
assert not submission["event_id"].duplicated().any()
assert submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].ge(0.0).all().all()
assert submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].le(1.0).all().all()
assert (submission["prob_12h"] <= submission["prob_24h"]).all()
assert (submission["prob_24h"] <= submission["prob_48h"]).all()
assert (submission["prob_48h"] <= submission["prob_72h"]).all()

output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
submission.to_csv(os.path.join(output_dir, "submission.csv"), index=False)

print("DATA_DIR:", DATA_DIR)
print("archives_used:", len(archives))
print(submission.head())
print("saved:", os.path.join(output_dir, "submission.csv"))


DATA_DIR: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
archives_used: 0
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.005393  0.006549  0.007200       1.0
1  13353600  0.387382  0.838893  0.951416       1.0
2  13942327  0.005393  0.006549  0.012738       1.0
3  16112781  0.664584  0.915582  0.956112       1.0
4  17132808  0.022172  0.026923  0.026923       1.0
saved: /kaggle/working/submission.csv
